In [ ]:
!pip install -q langchain langchain-community
!pip install -q sentence-transformers faiss-cpu rank_bm25
!pip install -q pypdf gradio crewai groq

In [ ]:
import os
import numpy as np
import faiss
import gradio as gr

from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from langchain.text_splitter import RecursiveCharacterTextSplitter
from crewai import Agent, Task, Crew
from groq import Groq

In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

def call_llm(prompt):
    response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

In [ ]:
def load_and_chunk_pdf(pdf_file):

    reader = PdfReader(pdf_file)
    text = ""

    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = splitter.split_text(text)

    return chunks

In [ ]:
def create_dense_index(chunks):

    embeddings = embedding_model.encode(chunks)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))

    return index, embeddings

In [ ]:
def create_sparse_index(chunks):

    tokenized_chunks = [chunk.split(" ") for chunk in chunks]
    bm25 = BM25Okapi(tokenized_chunks)

    return bm25

In [ ]:
def hybrid_search(query, chunks, faiss_index, embeddings, bm25, top_k=5):

    # Dense search
    query_embedding = embedding_model.encode([query])
    D, I = faiss_index.search(np.array(query_embedding), top_k)
    dense_results = [chunks[i] for i in I[0]]

    # Sparse search
    tokenized_query = query.split(" ")
    sparse_scores = bm25.get_scores(tokenized_query)
    sparse_top_indices = np.argsort(sparse_scores)[-top_k:]
    sparse_results = [chunks[i] for i in sparse_top_indices]

    # Fusion
    combined = list(set(dense_results + sparse_results))

    return combined

In [ ]:
summarizer_agent = Agent(
    role="Senior Document Intelligence Analyst",
    goal="Generate structured executive summaries from retrieved context",
    backstory="Expert in enterprise document analysis and structured reporting.",
    verbose=False
)

In [ ]:
def generate_summary(context_chunks, query):

    context = "\n\n".join(context_chunks)

    prompt = f"""
    You are an enterprise document summarization AI.

    Based on the retrieved context below:

    {context}

    Provide:
    1. Executive Summary
    2. Key Insights
    3. Critical Risks
    4. Recommendations
    5. Bullet-Point Highlights

    User Query: {query}

    Use structured professional formatting.
    """

    return call_llm(prompt)

In [ ]:
def process_pdf(pdf_file, query):

    chunks = load_and_chunk_pdf(pdf_file)

    faiss_index, embeddings = create_dense_index(chunks)
    bm25 = create_sparse_index(chunks)

    retrieved_chunks = hybrid_search(
        query, chunks, faiss_index, embeddings, bm25
    )

    summary = generate_summary(retrieved_chunks, query)

    return summary

In [ ]:
custom_css = """
body {
    background-color: #f4f7fb;
}
.header {
    text-align: center;
    font-size: 24px;
    font-weight: bold;
    color: #0b3d91;
    margin-bottom: 10px;
}
.footer {
    text-align: center;
    color: gray;
    font-size: 12px;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("<div class='header'>Hybrid RAG PDF Summarizer</div>")

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF Document")
        query_input = gr.Textbox(
            label="Enter Summary Instruction",
            placeholder="Summarize financial overview and risks..."
        )

    run_button = gr.Button("Generate Structured Summary", variant="primary")
    output_box = gr.Markdown()

    run_button.click(
        process_pdf,
        inputs=[pdf_input, query_input],
        outputs=output_box
    )

    gr.Markdown("<div class='footer'>Powered by Hybrid RAG | Dense + Sparse Retrieval | CrewAI Orchestration</div>")

demo.launch(debug=True)